In [0]:
# =========================================================
# SILVER LAYER: IMPORTS AND CONFIGURATION
# =========================================================

from datetime import datetime, timezone
import uuid

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)
from delta.tables import DeltaTable


# =========================================================
# TABLE CONFIGURATION
# =========================================================

bronze_table = "bronze.world_bank_food_prices"
silver_table = "silver.world_bank_food_prices"
reject_table = "rejects.silver_food_price_rejects"

run_id = str(uuid.uuid4())

pipeline_start = datetime.now(timezone.utc).replace(tzinfo=None)

print("Silver Run ID:", run_id)
print("Bronze source:", bronze_table)
print("Silver target:", silver_table)


# =========================================================
# CREATE REQUIRED SCHEMAS
# =========================================================

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS rejects")
spark.sql("CREATE SCHEMA IF NOT EXISTS audit")

In [0]:
# =========================================================
# READ BRONZE TABLE
# =========================================================

if not spark.catalog.tableExists(bronze_table):
    raise ValueError(
        f"Bronze table does not exist: {bronze_table}"
    )

bronze_df = spark.table(bronze_table)

print("Bronze row count:", bronze_df.count())
print("Bronze column count:", len(bronze_df.columns))

# Display actual column names and data types
bronze_df.printSchema()

# Display a sample of the data
display(bronze_df.limit(10))

In [0]:
# =========================================================
# SILVER TRANSFORMATION
# =========================================================

source_columns = bronze_df.columns

required_columns = ["ISO3", "geo_id", "price_date"]

missing_columns = [
    column
    for column in required_columns
    if column not in source_columns
]

if missing_columns:
    raise ValueError(
        f"Missing required business key columns: {missing_columns}"
    )


# ---------------------------------------------------------
# REMOVE UNWANTED WHITESPACE AND NORMALIZE EMPTY STRINGS
# ---------------------------------------------------------

silver_df = bronze_df

for column in source_columns:

    if column not in [
        "_ingestion_timestamp",
        "load_timestamp"
    ]:
        silver_df = silver_df.withColumn(
            column,
            F.when(
                F.trim(F.col(column).cast("string")) == "",
                F.lit(None)
            ).otherwise(
                F.trim(F.col(column).cast("string"))
            )
        )


# ---------------------------------------------------------
# STANDARDIZE BUSINESS KEYS
# ---------------------------------------------------------

silver_df = (
    silver_df
    .withColumn("ISO3", F.upper(F.trim(F.col("ISO3"))))
    .withColumn("geo_id", F.trim(F.col("geo_id")))
    .withColumn(
        "price_date",
        F.to_date(F.col("price_date"))
    )
)


# ---------------------------------------------------------
# EXPLICIT NUMERIC TYPE CONVERSIONS
# ---------------------------------------------------------

double_columns = [
    "mp_price",
    "lat",
    "lon"
]

integer_columns = [
    "mp_year",
    "mp_month"
]

for column in double_columns:

    if column in silver_df.columns:
        silver_df = silver_df.withColumn(
            column,
            F.expr(
                f"try_cast(`{column}` AS DOUBLE)"
            )
        )

for column in integer_columns:

    if column in silver_df.columns:
        silver_df = silver_df.withColumn(
            column,
            F.expr(
                f"try_cast(`{column}` AS INT)"
            )
        )


# ---------------------------------------------------------
# ADD SILVER METADATA
# ---------------------------------------------------------

silver_df = (
    silver_df
    .withColumn("_silver_run_id", F.lit(run_id))
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

print("Initial Silver transformation completed.")